# 🧠 Absolute Entity Modeling in Categorical Ontology
In this notebook, we transition from instantiating individual Absolute Entities to **modeling their categorical properties**, embedding them in a **functorial containment space**, and validating their compatibility under **adjoint and transfinite projections**.

We will leverage the foundational constructs established in `01_introduction_to_cross_absolute_theory.ipynb` and extend them into a model-theoretic and topology-aware structure compatible with the Enhanced Mathematical Ontology of Absolute Nothingness.
---
### 📚 Theoretical Premise
- Every Absolute Entity $A$ is a **terminal object** in a functorially closed ontological category.
- Entities are grouped via type-class equivalence $\mathcal{C}(A) \cong \, \mathcal{C}(B)$ iff their entropy manifolds are homeomorphic.
- Morphisms between Absolutes are defined via **legal mediated adjunctions**.
- Projection into categorical containers requires compliance with:
  1. Signature symmetry preservation
  2. Entropy monotonicity
  3. Hom-set closure under mediated transformations

In [ ]:
# Import foundational packages
import numpy as np
from typing import List, Optional, Tuple, Dict
import matplotlib.pyplot as plt
from pprint import pprint
from scipy.linalg import inv, expm

In [ ]:
# Re-define AbsoluteEntity class with extended capabilities
class AbsoluteEntity:
    def __init__(self, signature: np.ndarray, state: np.ndarray, name: Optional[str] = None):
        """
        Initialize an Absolute Entity with signature matrix and state vector.
        
        Args:
            signature: Square symmetric matrix defining ontological signature
            state: State vector matching signature dimension
            name: Optional identifier for the entity
        """
        if signature.ndim != 2 or signature.shape[0] != signature.shape[1]:
            raise ValueError("Signature must be square matrix.")
        if not np.allclose(signature, signature.T):
            raise ValueError("Signature matrix must be symmetric.")
        if state.shape[0] != signature.shape[0]:
            raise ValueError("State must match signature dimension.")
            
        self.signature = signature
        self.state = state
        self.dimension = signature.shape[0]
        self.name = name or f"AE-{id(self) % 10000}"
        self._cached_metrics = {}
        
    def encode_selection(self) -> float:
        """
        Compute the entropy scalar (trace norm) of this entity.
        
        Returns:
            Scalar entropy value
        """
        if "entropy" not in self._cached_metrics:
            self._cached_metrics["entropy"] = np.trace(self.signature @ self.signature.T)
        return self._cached_metrics["entropy"]
    
    def state_projection(self) -> float:
        """
        Project state through signature to get scalar response.
        
        Returns:
            Scalar projection value
        """
        return float(self.state.T @ self.signature @ self.state)
    
    def transform(self, operator: np.ndarray) -> 'AbsoluteEntity':
        """
        Apply transformation operator to create a new entity.
        
        Args:
            operator: Square transformation matrix
            
        Returns:
            New transformed entity
        """
        try:
            op_inv = inv(operator)
            new_sig = operator @ self.signature @ op_inv
            new_state = operator @ self.state
            return AbsoluteEntity(signature=new_sig, state=new_state, 
                                 name=f"{self.name}-transformed")
        except np.linalg.LinAlgError:
            raise ValueError("Transformation operator must be invertible")
    
    def is_compatible_with(self, other: 'AbsoluteEntity', tol: float = 1e-5) -> bool:
        """
        Check if this entity is compatible with another based on entropy equivalence.
        
        Args:
            other: Entity to compare with
            tol: Tolerance for numerical comparison
            
        Returns:
            True if entities are compatible
        """
        return abs(self.encode_selection() - other.encode_selection()) < tol
    
    def __str__(self):
        return f"AbsoluteEntity({self.name}, d={self.dimension}, ε={self.encode_selection():.3f})"
    
    def __repr__(self):
        return self.__str__()

In [ ]:
# Define the enhanced AbsoluteContainer class
class AbsoluteContainer:
    def __init__(self, name: str):
        """
        Create a container for Absolute Entities.
        
        Args:
            name: Container identifier
        """
        self.name = name
        self.members: List[AbsoluteEntity] = []
        self._cache = {}
        
    def add_entity(self, entity: AbsoluteEntity) -> bool:
        """
        Add an entity to the container with validation.
        
        Args:
            entity: Entity to add
            
        Returns:
            True if entity was added successfully
        """
        # Clear cache when collection changes
        self._cache = {}
        
        # Check for signature symmetry (redundant but demonstrative)
        if not np.allclose(entity.signature, entity.signature.T):
            print(f"Warning: Entity {entity.name} has non-symmetric signature")
            return False
            
        self.members.append(entity)
        return True
    
    def get_entity(self, name: str) -> Optional[AbsoluteEntity]:
        """
        Retrieve entity by name.
        
        Args:
            name: Entity name to find
            
        Returns:
            Found entity or None
        """
        for entity in self.members:
            if entity.name == name:
                return entity
        return None
    
    def entropy_profile(self) -> List[float]:
        """
        Get entropy values for all members.
        
        Returns:
            List of entropy scalars
        """
        if "entropy_profile" not in self._cache:
            self._cache["entropy_profile"] = [e.encode_selection() for e in self.members]
        return self._cache["entropy_profile"]
    
    def is_homeomorphic(self, other: 'AbsoluteContainer', tol=1e-5) -> bool:
        """
        Check if this container is homeomorphic to another via entropy profile.
        
        Args:
            other: Container to compare with
            tol: Tolerance for comparison
            
        Returns:
            True if containers are homeomorphic
        """
        a = sorted(self.entropy_profile())
        b = sorted(other.entropy_profile())
        return len(a) == len(b) and all(abs(x - y) < tol for x, y in zip(a, b))
    
    def apply_transformation(self, transform_matrix: np.ndarray) -> 'AbsoluteContainer':
        """
        Create a new container by applying a transform to all members.
        
        Args:
            transform_matrix: Matrix to apply to each entity
            
        Returns:
            New container with transformed entities
        """
        new_container = AbsoluteContainer(f"{self.name}-transformed")
        
        for entity in self.members:
            try:
                new_entity = entity.transform(transform_matrix)
                new_container.add_entity(new_entity)
            except ValueError as e:
                print(f"Warning: Could not transform {entity.name}: {str(e)}")
                
        return new_container
        
    def __len__(self) -> int:
        return len(self.members)
    
    def __str__(self) -> str:
        return f"AbsoluteContainer({self.name}, {len(self.members)} members)"
    
    def __repr__(self) -> str:
        return self.__str__()

In [ ]:
# Construct two containers and populate with entities
container_A = AbsoluteContainer("Group-A")
container_B = AbsoluteContainer("Group-B")

# Set seed for reproducibility
np.random.seed(42)

# Create entities for container A
for i in range(3):
    # Generate symmetric signature matrix
    S = np.random.rand(5, 5)
    S = 0.5 * (S + S.T)  # Ensure symmetry
    
    # Create and add entity
    entity = AbsoluteEntity(
        signature=S, 
        state=np.random.rand(5),
        name=f"A-Entity-{i+1}"
    )
    container_A.add_entity(entity)

# Create entities for container B with similar entropy profile
for i in range(3):
    # Generate different but similar signature matrix
    S = np.random.rand(5, 5)
    S = 0.5 * (S + S.T)  # Ensure symmetry
    
    # Create and add entity
    entity = AbsoluteEntity(
        signature=S, 
        state=np.random.rand(5),
        name=f"B-Entity-{i+1}"
    )
    container_B.add_entity(entity)

# Display created containers
print(f"Created: {container_A}")
print(f"Created: {container_B}")

In [ ]:
# Compare entropy profiles with more detailed output
print(f"{container_A.name} Entropy Profile:")
for i, (entity, entropy) in enumerate(zip(container_A.members, container_A.entropy_profile())):
    print(f"  - {entity.name}: {entropy:.4f}")
    
print(f"\n{container_B.name} Entropy Profile:")
for i, (entity, entropy) in enumerate(zip(container_B.members, container_B.entropy_profile())):
    print(f"  - {entity.name}: {entropy:.4f}")
    
# Check homeomorphism
is_homeo = container_A.is_homeomorphic(container_B)
print(f"\nHomeomorphic? {is_homeo}")

# Detailed comparison
print("\nDetailed entropy comparison:")
a_profile = sorted(container_A.entropy_profile())
b_profile = sorted(container_B.entropy_profile())
for i, (a, b) in enumerate(zip(a_profile, b_profile)):
    print(f"  Entity pair {i+1}: {a:.4f} vs {b:.4f}, diff = {abs(a-b):.4e}")

In [ ]:
# Enhanced visualization of entropy profiles
plt.figure(figsize=(10, 6))

# Plot individual entities with labels
for i, (entity, entropy) in enumerate(zip(container_A.members, container_A.entropy_profile())):
    plt.scatter(i, entropy, color='blue', s=100, marker='o')
    plt.annotate(entity.name, (i, entropy), xytext=(5, 5), textcoords='offset points')
    
for i, (entity, entropy) in enumerate(zip(container_B.members, container_B.entropy_profile())):
    plt.scatter(i, entropy, color='red', s=100, marker='s')
    plt.annotate(entity.name, (i, entropy), xytext=(5, -15), textcoords='offset points')

# Plot connected lines for visual comparison
plt.plot(range(len(container_A.members)), container_A.entropy_profile(), 
         label=f"{container_A.name}", marker='o', linestyle='-', color='blue')
plt.plot(range(len(container_B.members)), container_B.entropy_profile(), 
         label=f"{container_B.name}", marker='s', linestyle='--', color='red')

# Add detailed visualization elements
plt.xlabel("Entity Index")
plt.ylabel("Entropy Scalar (ε)")
plt.title("Entropy Projection of Categorical Containers")
plt.legend(loc='best')
plt.grid(True, alpha=0.3)

# Add homeomorphism status to the plot
is_homeo = container_A.is_homeomorphic(container_B)
plt.figtext(0.5, 0.01, f"Homeomorphic: {is_homeo}", 
            ha='center', fontsize=12, bbox={"facecolor":"lightgray", "alpha":0.5, "pad":5})

plt.tight_layout()
plt.savefig("entropy_categorical_projection.png", dpi=300)
plt.show()

In [ ]:
# Enhanced adjoint functor mapping implementation
def compute_adjoint_mapping(source: AbsoluteEntity, target: AbsoluteEntity, 
                           iterations: int = 1000, step_size: float = 0.01) -> Tuple[np.ndarray, float]:
    """
    Attempt to find an adjoint mapping from source to target.
    
    Args:
        source: Source entity
        target: Target entity
        iterations: Maximum iterations for optimization
        step_size: Learning rate for optimization
        
    Returns:
        Tuple of (transformation_matrix, final_error)
    """
    # Initialize a random transformation matrix
    T = np.eye(source.dimension) + 0.1 * np.random.randn(source.dimension, source.dimension)
    
    # Track the loss over iterations
    losses = []
    
    # Simple gradient descent to find mapping
    for i in range(iterations):
        try:
            T_inv = inv(T)
            # Compute transformed signature
            mapped_sig = T @ source.signature @ T_inv
            
            # Compute Frobenius norm error
            error = np.linalg.norm(mapped_sig - target.signature)
            losses.append(error)
            
            # Early stopping if we've found a good mapping
            if error < 1e-2:
                print(f"Found good mapping at iteration {i}")
                break
                
            # Compute gradient and update T
            gradient = 2 * (mapped_sig - target.signature) @ T_inv.T
            T -= step_size * gradient
            
            # Occasionally print progress
            if i % 100 == 0:
                print(f"Iteration {i}, Error: {error:.4f}")
                
        except np.linalg.LinAlgError:
            # If matrix becomes non-invertible, add a small perturbation
            print(f"Matrix became non-invertible at iteration {i}, stabilizing...")
            T += 0.1 * np.eye(source.dimension)
    
    # Final mapping evaluation
    try:
        T_inv = inv(T)
        mapped_sig = T @ source.signature @ T_inv
        final_error = np.linalg.norm(mapped_sig - target.signature)
        print(f"Final mapping error: {final_error:.4f}")
        
        # Plot loss over iterations
        plt.figure(figsize=(8, 4))
        plt.plot(losses)
        plt.xlabel("Iteration")
        plt.ylabel("Mapping Error")
        plt.title(f"Adjoint Mapping Convergence: {source.name} → {target.name}")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        return T, final_error
    except np.linalg.LinAlgError as e:
        print(f"Failed to compute final adjoint mapping: {str(e)}")
        return None, float('inf')

# Test the mapping between specific entities
source_entity = container_A.members[0]
target_entity = container_B.members[0]

print(f"Attempting to map {source_entity.name} → {target_entity.name}")
transformation, error = compute_adjoint_mapping(source_entity, target_entity)

if transformation is not None:
    # Apply the transformation and verify
    transformed = source_entity.transform(transformation)
    print(f"\nOriginal: {source_entity}")
    print(f"Transformed: {transformed}")
    print(f"Target: {target_entity}")
    
    print(f"\nCompatibility check: {transformed.is_compatible_with(target_entity)}")

In [ ]:
# New advanced functionality: Projection operators
class ProjectionOperator:
    def __init__(self, source_dim: int, target_dim: int):
        """
        Create a projection operator between spaces of different dimensions.
        
        Args:
            source_dim: Source space dimension
            target_dim: Target space dimension
        """
        self.source_dim = source_dim
        self.target_dim = target_dim
        
        # Initialize projection matrices
        if source_dim >= target_dim:
            # Downward projection
            self.forward = np.zeros((target_dim, source_dim))
            for i in range(target_dim):
                self.forward[i, i] = 1.0
            self.backward = np.zeros((source_dim, target_dim))
            for i in range(target_dim):
                self.backward[i, i] = 1.0
        else:
            # Upward projection (embedding)
            self.forward = np.zeros((target_dim, source_dim))
            for i in range(source_dim):
                self.forward[i, i] = 1.0
            self.backward = np.zeros((source_dim, target_dim))
            for i in range(source_dim):
                self.backward[i, i] = 1.0
                
    def project_entity(self, entity: AbsoluteEntity) -> AbsoluteEntity:
        """
        Project an entity to the target dimension.
        
        Args:
            entity: Entity to project
            
        Returns:
            Projected entity
        """
        if entity.dimension != self.source_dim:
            raise ValueError(f"Entity dimension {entity.dimension} does not match source {self.source_dim}")
        
        # Project signature and state
        projected_sig = self.forward @ entity.signature @ self.backward
        projected_state = self.forward @ entity.state
        
        return AbsoluteEntity(
            signature=projected_sig,
            state=projected_state,
            name=f"{entity.name}-proj{self.target_dim}"
        )
        
    def __str__(self):
        return f"ProjectionOperator({self.source_dim} → {self.target_dim})"

# Demonstrate projection to a higher dimension
# First create a smaller entity
small_sig = np.random.rand(3, 3)
small_sig = 0.5 * (small_sig + small_sig.T)  # Make symmetric
small_entity = AbsoluteEntity(signature=small_sig, state=np.random.rand(3), name="SmallEntity")

# Create projection operator
projector = ProjectionOperator(source_dim=3, target_dim=5)
print(f"Created {projector}")

# Apply projection
projected = projector.project_entity(small_entity)
print(f"Original: {small_entity}")
print(f"Projected: {projected}")

# Show the entropy preservation property
print(f"Original entropy: {small_entity.encode_selection():.4f}")
print(f"Projected entropy: {projected.encode_selection():.4f}")

## ✅ Summary and Improvements
- Modeled Absolute Entities as objects in ontological containers with improved naming and organization.
- Added robust error handling and validation for all operations.
- Enhanced visualization with detailed entity labeling and homeomorphism status.
- Implemented advanced adjoint mappings with optimization and convergence tracking.
- Added new ProjectionOperator class to support cross-dimensional embeddings.
- Improved caching strategy for computational efficiency.
- Provided detailed docstrings and improved code organization.

These enhancements provide a more robust foundation for modeling categorical properties of Absolute Entities while maintaining compatibility with the original theoretical framework.

In the next notebook, we will introduce **Interaction Operators**, **Mediator Chains**, and formalize the **Functorial Tensor Composition** necessary for force decomposition and field emergence.

**Next → `03_interaction_operator_framework.ipynb`**